<a href="https://colab.research.google.com/github/kuatovakamila/irrigation_prediction_need_model/blob/main/irrigation_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e4')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/playground-series-s6e4


In [ ]:
import kagglehub
path = kagglehub.dataset_download("miadul/irrigation-water-requirement-prediction-dataset")
print("Path:", path)

import os
print(os.listdir(path))

df_orig = pd.read_csv(os.path.join(path, 'irrigation_prediction.csv'))
print(df_orig.shape)
print(df_orig.columns.tolist())

100%|██████████| 342k/342k [00:00<00:00, 901kB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/miadul/irrigation-water-requirement-prediction-dataset/versions/1
['irrigation_prediction.csv']
(10000, 20)
['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']


In [ ]:
# ============ CELL 1: Imports and data loading ============
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from itertools import combinations

# Load competition data
df_train = pd.read_csv('/root/.cache/kagglehub/competitions/playground-series-s6e4/train.csv')
df_test = pd.read_csv('/root/.cache/kagglehub/competitions/playground-series-s6e4/test.csv')

# Load original dataset (download "irrigation-water-requirement-prediction-dataset"
# from Kaggle datasets first, upload to Colab as irrigation_prediction.csv)
df_orig = pd.read_csv('/root/.cache/kagglehub/datasets/miadul/irrigation-water-requirement-prediction-dataset/versions/1/irrigation_prediction.csv')

le = LabelEncoder()
df_train["Irrigation_Need"] = le.fit_transform(df_train["Irrigation_Need"])
df_orig["Irrigation_Need"] = le.transform(df_orig["Irrigation_Need"])

In [ ]:
# ============ CELL 2: Feature engineering ============
cat_cols = ["Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
            "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"]

X = df_train.drop(["id", "Irrigation_Need"], axis=1)
Y = df_train["Irrigation_Need"]
X_orig = df_orig.drop(["Irrigation_Need"], axis=1)
Y_orig = df_orig["Irrigation_Need"]
X_test = df_test.drop(["id"], axis=1)

# Convert categoricals to strings
for col in cat_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    X_orig[col] = X_orig[col].astype(str)

# Combine train + original
X = pd.concat([X, X_orig], axis=0).reset_index(drop=True)
Y = pd.concat([Y, Y_orig], axis=0).reset_index(drop=True)

# Threshold binary features
for df in [X, X_test]:
    df['soil_lt_25']  = (df['Soil_Moisture']  < 25).astype(int)
    df['wind_gt_10']  = (df['Wind_Speed_kmh'] > 10).astype(int)
    df['temp_gt_30']  = (df['Temperature_C']  > 30).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm']    < 300).astype(int)

# Bigram interactions
top_cats = ['Crop_Growth_Stage', 'Mulching_Used', 'Soil_Type', 'Crop_Type']
bigram_cols = []
for c1, c2 in combinations(top_cats, 2):
    name = f'{c1}_x_{c2}'
    X[name] = X[c1].astype(str) + '_' + X[c2].astype(str)
    X_test[name] = X_test[c1].astype(str) + '_' + X_test[c2].astype(str)
    bigram_cols.append(name)

# Digit features
num_targets = ['Soil_Moisture', 'Temperature_C', 'Wind_Speed_kmh', 'Rainfall_mm']
digit_cols = []
for c in num_targets:
    for k in range(-2, 3):
        name = f'{c}_d{k}'
        X[name] = ((X[c] // (10**k)) % 10).astype('int8')
        X_test[name] = ((X_test[c] // (10**k)) % 10).astype('int8')
        digit_cols.append(name)

cb_class_weights = [9.994534239087999, 0.5677065453903559, 0.8783704033092605]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Feature engineering done. Shape:", X.shape)

Feature engineering done. Shape: (640000, 49)


In [ ]:
# ============ CELL 3: Helper function ============
def add_fold_aggregates(X_train, X_val, X_test, cat_cols, num_cols):
    X_train_copy = X_train.copy()
    X_val_copy = X_val.copy()
    X_test_copy = X_test.copy()

    new_cols = []
    for cat in cat_cols:
        for num in num_cols:
            means = X_train.groupby(cat)[num].mean()
            global_mean = X_train_copy[num].mean()
            name_mean = f'{num}_mean_by_{cat}'
            name_diff = f'{num}_diff_by_{cat}'

            X_train_copy[name_mean] = X_train_copy[cat].map(means).fillna(global_mean)
            X_val_copy[name_mean] = X_val_copy[cat].map(means).fillna(global_mean)
            X_test_copy[name_mean] = X_test_copy[cat].map(means).fillna(global_mean)

            X_train_copy[name_diff] = X_train_copy[num] - X_train_copy[name_mean]
            X_val_copy[name_diff]   = X_val_copy[num] - X_val_copy[name_mean]
            X_test_copy[name_diff]  = X_test_copy[num] - X_test_copy[name_mean]

            new_cols.extend([name_mean, name_diff])
    return X_train_copy, X_val_copy, X_test_copy, new_cols

In [ ]:
# ============ CELL 4: CatBoost - 3 seeds × 5 folds = 15 models (~30 min on GPU) ============
cb_oof_seeds = np.zeros((len(X), 3))
cb_test_seeds = np.zeros((len(X_test), 3))
cb_drop = bigram_cols + digit_cols

for seed in [42, 123, 999]:
    print(f"=== CatBoost seed {seed} ===")
    cb_oof_tmp = np.zeros((len(X), 3))
    cb_test_tmp = np.zeros((len(X_test), 3))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, Y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx]

        X_train_cb = X_train.drop(columns=cb_drop).copy()
        X_val_cb = X_val.drop(columns=cb_drop).copy()
        X_test_cb = X_test.drop(columns=cb_drop).copy()

        cb_cat_idx = [X_train_cb.columns.get_loc(c) for c in cat_cols]

        model_cb = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.09682530687769166,
            depth=4, l2_leaf_reg=6.59689095776599,
            bagging_temperature=0.527738847069491,
            random_strength=0.7128283543743552,
            loss_function='MultiClass',
            class_weights=cb_class_weights,
            cat_features=cb_cat_idx,
            random_seed=seed,
            early_stopping_rounds=50,
            task_type="GPU",
            verbose=0
        )
        model_cb.fit(X_train_cb, Y_train, eval_set=[(X_val_cb, Y_val)])
        cb_oof_tmp[val_idx] = model_cb.predict_proba(X_val_cb)
        cb_test_tmp += model_cb.predict_proba(X_test_cb) / 5
        print(f"  Fold {fold+1} done")

    cb_oof_seeds += cb_oof_tmp / 3
    cb_test_seeds += cb_test_tmp / 3

print(f"CB OOF: {balanced_accuracy_score(Y, np.argmax(cb_oof_seeds, axis=1)):.4f}")

=== CatBoost seed 42 ===
  Fold 1 done
  Fold 2 done
  Fold 3 done
  Fold 4 done
  Fold 5 done
=== CatBoost seed 123 ===
  Fold 1 done
  Fold 2 done
  Fold 3 done
  Fold 4 done
  Fold 5 done
=== CatBoost seed 999 ===
  Fold 1 done
  Fold 2 done
  Fold 3 done
  Fold 4 done
  Fold 5 done
CB OOF: 0.9714


In [ ]:
# ============ CELL 5: XGBoost - 3 seeds × 5 folds = 15 models (~1.5 hr on GPU) ============
cat_features_extended = cat_cols + bigram_cols + digit_cols
num_for_aggs = ['Soil_Moisture', 'Temperature_C', 'Wind_Speed_kmh', 'Rainfall_mm']

xgb_oof_seeds = np.zeros((len(X), 3))
xgb_test_seeds = np.zeros((len(X_test), 3))

for seed in [42, 123, 999]:
    print(f"=== XGBoost seed {seed} ===")
    xgb_oof_tmp = np.zeros((len(X), 3))
    xgb_test_tmp = np.zeros((len(X_test), 3))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, Y)):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx]
        X_test_fold = X_test.copy()

        X_train, X_val, X_test_fold, _ = add_fold_aggregates(
            X_train, X_val, X_test_fold, cat_cols, num_for_aggs
        )

        encoder = TargetEncoder(target_type='multiclass', smooth='auto', cv=5, random_state=42)
        te_train = encoder.fit_transform(X_train[cat_features_extended], Y_train)
        te_val = encoder.transform(X_val[cat_features_extended])
        te_test = encoder.transform(X_test_fold[cat_features_extended])

        te_cols = [f"{c}_te{k}" for c in cat_features_extended for k in range(3)]

        X_train_xgb = X_train.drop(columns=cat_features_extended).copy()
        X_val_xgb = X_val.drop(columns=cat_features_extended).copy()
        X_test_xgb = X_test_fold.drop(columns=cat_features_extended).copy()

        X_train_xgb[te_cols] = te_train
        X_val_xgb[te_cols] = te_val
        X_test_xgb[te_cols] = te_test

        sample_weights = compute_sample_weight(class_weight='balanced', y=Y_train)

        model_xgb = XGBClassifier(
            objective='multi:softprob', num_class=3,
            tree_method='hist', device='cuda',
            learning_rate=0.05, max_depth=3, n_estimators=2000,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, early_stopping_rounds=50,
            eval_metric='mlogloss', verbosity=0
        )
        model_xgb.fit(X_train_xgb, Y_train,
                      sample_weight=sample_weights,
                      eval_set=[(X_val_xgb, Y_val)], verbose=False)
        xgb_oof_tmp[val_idx] = model_xgb.predict_proba(X_val_xgb)
        xgb_test_tmp += model_xgb.predict_proba(X_test_xgb) / 5
        print(f"  Fold {fold+1} done")

    xgb_oof_seeds += xgb_oof_tmp / 3
    xgb_test_seeds += xgb_test_tmp / 3

print(f"XGB OOF: {balanced_accuracy_score(Y, np.argmax(xgb_oof_seeds, axis=1)):.4f}")

=== XGBoost seed 42 ===


/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

  Fold 1 done


/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

  Fold 2 done


...

  Fold 3 done


/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

  Fold 4 done


/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_xgb[te_cols] = te_train
/tmp/ipykernel_747/3161938554.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

  Fold 5 done
XGB OOF: 0.9737


In [ ]:
# ============ CELL 6: LightGBM - 5 folds (~30 min on CPU) ============
lgb_oof = np.zeros((len(X), 3))
lgb_test = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, Y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx]
    X_test_fold = X_test.copy()

    X_train, X_val, X_test_fold, _ = add_fold_aggregates(
        X_train, X_val, X_test_fold, cat_cols, num_for_aggs
    )

    te = TargetEncoder(target_type='multiclass', smooth='auto', cv=5, random_state=42)
    te_train = te.fit_transform(X_train[cat_features_extended], Y_train)
    te_val = te.transform(X_val[cat_features_extended])
    te_test = te.transform(X_test_fold[cat_features_extended])

    te_cols = [f"{c}_te{k}" for c in cat_features_extended for k in range(3)]

    X_train_lgb = X_train.drop(columns=cat_features_extended)
    X_val_lgb = X_val.drop(columns=cat_features_extended)
    X_test_lgb = X_test_fold.drop(columns=cat_features_extended)

    X_train_lgb = pd.concat([X_train_lgb, pd.DataFrame(te_train, columns=te_cols, index=X_train_lgb.index)], axis=1)
    X_val_lgb = pd.concat([X_val_lgb, pd.DataFrame(te_val, columns=te_cols, index=X_val_lgb.index)], axis=1)
    X_test_lgb = pd.concat([X_test_lgb, pd.DataFrame(te_test, columns=te_cols, index=X_test_lgb.index)], axis=1)

    model_lgb = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.05, num_leaves=15,
        min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, class_weight='balanced',
        random_state=42, verbose=-1
    )
    model_lgb.fit(X_train_lgb, Y_train,
                  eval_set=[(X_val_lgb, Y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    lgb_oof[val_idx] = model_lgb.predict_proba(X_val_lgb)
    lgb_test += model_lgb.predict_proba(X_test_lgb) / 5
    print(f"LGB Fold {fold+1} done")

print(f"LGB OOF: {balanced_accuracy_score(Y, np.argmax(lgb_oof, axis=1)):.4f}")

LGB Fold 1 done
LGB Fold 2 done
LGB Fold 3 done
LGB Fold 4 done
LGB Fold 5 done
LGB OOF: 0.9712


In [ ]:
# ============ CELL 7: Stack with Logistic Regression meta-model ============
meta_X_train = np.hstack([cb_oof_seeds, xgb_oof_seeds, lgb_oof])
meta_X_test = np.hstack([cb_test_seeds, xgb_test_seeds, lgb_test])

meta_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
meta_model.fit(meta_X_train, Y)

stack_oof = meta_model.predict_proba(meta_X_train)
print(f"Stacked OOF: {balanced_accuracy_score(Y, np.argmax(stack_oof, axis=1)):.4f}")

test_proba = meta_model.predict_proba(meta_X_test)
test_classes = np.argmax(test_proba, axis=1)

submission = pd.DataFrame({
    "id": df_test["id"],
    "Irrigation_Need": le.inverse_transform(test_classes)
})
submission.to_csv("submission_stacked.csv", index=False)
print(submission["Irrigation_Need"].value_counts())

from google.colab import files
files.download('submission_stacked.csv')